In [2]:
import pandas as pd

# **1.Data Understanding & Loading CSV file**

**Encoding the Online Retail dataset which contains special character like £, UTF-8 gives a UnicodeDecodeError, so use encoding_error replace**

In [3]:
df = pd.read_csv(
    "dataset.csv",
    encoding_errors="replace"
)
df

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,40513.35139,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,40513.35139,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,40513.35139,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,40513.35139,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,40513.35139,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,40886.53472,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,40886.53472,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,40886.53472,4.15,12680.0,France
541908,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,40886.53472,4.95,12680.0,France


In [4]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,40513.35139,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,40513.35139,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,40513.35139,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,40513.35139,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,40513.35139,3.39,17850.0,United Kingdom


**Shape and data types**

In [5]:
print("Rows, Columns:", df.shape)
df.info()

Rows, Columns: (541910, 8)
<class 'pandas.DataFrame'>
RangeIndex: 541910 entries, 0 to 541909
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Invoice      541910 non-null  str    
 1   StockCode    541910 non-null  str    
 2   Description  540456 non-null  str    
 3   Quantity     541910 non-null  int64  
 4   InvoiceDate  541910 non-null  float64
 5   Price        541910 non-null  float64
 6   Customer ID  406830 non-null  float64
 7   Country      541910 non-null  str    
dtypes: float64(3), int64(1), str(4)
memory usage: 33.1 MB


**Missing values and duplicates**

In [6]:
missing_values = df.isna().sum().to_frame("missing")
missing_values["percent"] = missing_values["missing"] / len(df) * 100

print(missing_values)
print(f"Fully duplicaated rows:", df.duplicated().sum())

             missing    percent
Invoice            0   0.000000
StockCode          0   0.000000
Description     1454   0.268310
Quantity           0   0.000000
InvoiceDate        0   0.000000
Price              0   0.000000
Customer ID   135080  24.926648
Country            0   0.000000
Fully duplicaated rows: 5268


**Statistical summary**

In [7]:
print(df[["Quantity", "Price"]].describe())

            Quantity          Price
count  541910.000000  541910.000000
mean        9.552234       4.611138
std       218.080957      96.759765
min    -80995.000000  -11062.060000
25%         1.000000       1.250000
50%         3.000000       2.080000
75%        10.000000       4.130000
max     80995.000000   38970.000000


**Size of the Online retail business**

**`.nunique()` counts the number of unique/different values. Simply it means How many different invoices/orders are in my dataset**

In [8]:
print("Invoices :", df["Invoice"].nunique())
print("Products :", df["StockCode"].nunique())
print("Customers :", df["Customer ID"].nunique())
print("Countries :", df["Country"].nunique())

start = pd.to_datetime("1899-12-30") + pd.to_timedelta(df["InvoiceDate"].min(), unit="D")
end = pd.to_datetime("1899-12-30") + pd.to_timedelta(df["InvoiceDate"].max(), unit="D")

print("Date Range:", start.date(), "to", end.date())

Invoices : 25900
Products : 4070
Customers : 4372
Countries : 38
Date Range: 2010-12-01 to 2011-12-09


**Finding Problem rows**

**`.match(r"^\d{5}")` This is called a regular expression (regex)
`r"..."` Treat this as a raw string. It's commonly used when writing regular expressions because it prevents Python from treating certain characters as escape characters.`^` It starts checking from the beginning of the StockCode. `\d` It means a digit from 0 to 9. `{5}` It means exactly 5 times. Altogether it means starting from the beginning, look for 5 digits**

In [9]:
print("Negative or zero quantity:", (df["Quantity"] <= 0).sum())
print("Zero or Negative price:", (df["Price"] <= 0).sum())

non_product = df[~df["StockCode"].astype(str).str.match(r"^\d{5}")]
print("non-product rows:", len(non_product))
print(non_product["StockCode"].value_counts().head(10))

Negative or zero quantity: 10624
Zero or Negative price: 2517
non-product rows: 2996
StockCode
POST            1257
DOT              710
M                571
C2               144
D                 77
S                 63
BANK CHARGES      37
AMAZONFEE         34
CRUK              16
DCGSSGIRL         13
Name: count, dtype: int64


# **2.Data Cleaning**

**Raw data -> rename columns -> fix dates -> remove duplicates -> remove invalid prices -> remove non-products -> clean text -> handle missing customers -> separate sales/returns -> remove extreme quantities -> summary**

**Copy and rename columns to snake_case**

**In `snake_case` We write everything in lowercase and if there are multiple words, we separate them with an underscore `_`**

In [10]:
data = df.copy()
data.columns = ["invoice", "stock_code", "description", "quantity", "invoice_date",
                "price", "customer_id", "country"]

log = [] # cleaning log

def record(step, before, after):
    log.append({"step": step, "rows_removed": before - after})
    

**Converting excel seriel date to a real datetime**

In [11]:
data["invoice_date"] = pd.to_datetime("1899-12-30") + pd.to_timedelta(data["invoice_date"], unit="D")
data["invoice_date"] = data["invoice_date"].dt.round("min")

**Removing exact duplicates**

In [12]:
before = len(data)
data = data.drop_duplicates()
record("Exact duplicate rows", before, len(data))

**Revoming price <= 0**

In [13]:
before = len(data)
data = data[data["price"] > 0]
record("Price <= 0 (adjustments, free items)", before, len(data))

**Removing non-product codes (POST, DOT, M, BANK CHARGES...)**

In [14]:
is_product = data["stock_code"].astype(str).str.match(r"^\d{5}")
before = len(data)
non_product_rows = data[~is_product] # kept aside just in case
data = data[is_product]
record("Non-product StockCodes", before, len(data))

**Missing descriptions + standarise text**

In [15]:
before = len(data)
data = data.dropna(subset=["description"])
record("Missing description", before, len(data))

data["description"] = data["description"].str.strip().str.upper()
data["stock_code"] = data["stock_code"].astype(str).str.strip().str.upper()
data["country"] = data["country"].str.strip()

**Missing Customer ID: keep rows, add a flag**

In [16]:
data["is_guest"] = data["customer_id"].isna()
data["customer_id"] = data["customer_id"].astype("Int64")

print("Rows without customer:", data["is_guest"].sum(), f"({data['is_guest'].mean():.1%})")

Rows without customer: 131516 (24.8%)


**Separate returns from sales**

In [17]:
returns = data[data["quantity"] < 0].copy()
sales = data[data["quantity"] > 0].copy()


**Outliners: remove huge orders that were cancelled, and their matching returns**

In [19]:
print(sales["quantity"].describe(percentiles=[.5, .9, .99, .999]))
print(sales.sort_values("quantity", ascending=False).head(5)
      [["invoice", "description", "quantity", "price", "invoice_date"]]
      )

before = len(sales) + len(returns)
sales = sales[sales["quantity"] < 10000]
returns = returns[returns["quantity"] > -10000]
record("Extreme quantity (>= 10,000) in sales and returns", before, len(sales) + len(returns))

count    522502.000000
mean         10.346506
std          37.855225
min           1.000000
50%           4.000000
90%          24.000000
99%         100.000000
99.9%       448.998000
max        4800.000000
Name: quantity, dtype: float64
       invoice                          description  quantity  price  \
421632  573008    WORLD WAR 2 GLIDERS ASSTD DESIGNS      4800   0.21   
206121  554868                 SMALL POPCORN HOLDER      4300   0.72   
97432   544612                EMPIRE DESIGN ROSETTE      3906   0.82   
270885  560599  ESSENTIAL BALM 3.5G TIN IN ENVELOPE      3186   0.06   
160546  550461   FAIRY CAKE FLANNEL ASSORTED COLOUR      3114   2.10   

              invoice_date  
421632 2011-10-27 12:26:00  
206121 2011-05-27 10:52:00  
97432  2011-02-22 10:43:00  
270885 2011-07-19 17:04:00  
160546 2011-04-18 13:20:00  


**cleaning summary**

In [20]:
log_df = pd.DataFrame(log)
log_df.loc[len(log_df)] = ["Total", log_df["rows_removed"].sum()]
print(log_df.to_string(index=False))
print("Raw rows :", len(df))
print("Clean sales :", len(sales))
print("Return rows :", len(returns))

                                             step  rows_removed
                             Exact duplicate rows          5268
             Price <= 0 (adjustments, free items)          2512
                           Non-product StockCodes          2958
                              Missing description             0
Extreme quantity (>= 10,000) in sales and returns             0
                                            Total         10738
Raw rows : 541910
Clean sales : 522502
Return rows : 8666
